In [1]:
# %pip install scikit-learn xgboost shap joblib fastapi uvicorn

In [2]:
import sklearn
import xgboost
import shap

print("sklearn:", sklearn.__version__)
print("xgboost:", xgboost.__version__)
print("shap:", shap.__version__)

c:\Users\aishw\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


sklearn: 1.9.0
xgboost: 3.3.0
shap: 0.52.0


In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"C:\\Users\\aishw\\SAR-Student-Success-Retention\\datasets\\student_success_dataset_30000.csv")

print(df.shape)
df.head()

(30000, 70)


,student_id,department,specialization,current_year,semester,gender,cgpa,semester_gpa,cgpa_trend,backlog_count,...,stress_score,time_management_score,consistency_score,academic_risk_band,dropout_risk_band,placement_risk_band,recommended_intervention,recommended_career_path,recommended_certification,recommended_course_track
0,STU202600001,Computer Engineering,Full Stack Development,Third Year,6,Male,9.00,9.15,Stable,1,...,62.52,61.18,75.33,Low,Low,Medium,Routine Academic Mentoring & Growth Tracking,IT Systems Consultant,Project Management Professional (PMP) Baseline,Academic Support & Catch-Up Track
1,STU202600002,Civil Engineering,Environmental Engineering,Second Year,3,Female,8.25,8.31,Stable,0,...,34.62,59.95,63.80,Low,Low,High,Routine Academic Mentoring & Growth Tracking,Structural & Infrastructure Planner,Project Management Professional (PMP) Baseline,Entrepreneurship Track
2,STU202600003,AI & ML,Computer Vision,Final Year,7,Female,8.81,8.79,Stable,0,...,34.43,74.90,91.29,Low,Low,Low,Routine Academic Mentoring & Growth Tracking,AI/ML Research Specialist,AWS Certified Machine Learning Specialty,Placement Fast Track
3,STU202600004,Electronics & Telecommunication,Embedded Systems,First Year,2,Male,8.83,8.69,Stable,0,...,32.98,72.21,88.28,Low,Low,High,Routine Academic Mentoring & Growth Tracking,Embedded Systems & IoT Engineer,PTC IoT Developer Certification,Academic Support & Catch-Up Track
4,STU202600005,Mechanical Engineering,CAD/CAM,Third Year,6,Male,9.34,9.36,Stable,0,...,45.71,74.01,87.37,Low,Low,Low,Routine Academic Mentoring & Growth Tracking,Product Design Engineer,Project Management Professional (PMP) Baseline,Placement Fast Track


In [4]:
target = "academic_risk_band"

In [5]:
drop_cols = [
    "student_id",

    "academic_risk_score",

    "academic_risk_band",

    "dropout_risk_band",

    "placement_risk_band",

    "placement_probability",

    "placement_readiness_score",

    "career_readiness_score"
]

X = df.drop(columns=drop_cols)

y = df[target]

print("Features:", X.shape)
print("Target:", y.shape)

Features: (30000, 62)
Target: (30000,)


In [6]:
print(y.value_counts())

print("\nPercentages:\n")

print((y.value_counts(normalize=True) * 100).round(2))

academic_risk_band
Low         24390
Medium       3754
High         1691
Critical      165
Name: count, dtype: int64

Percentages:

academic_risk_band
Low         81.30
Medium      12.51
High         5.64
Critical     0.55
Name: proportion, dtype: float64


In [7]:
# import sys
# print(sys.executable)

In [8]:
# !pip install scikit-learn

Create the Train/Test Split

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (24000, 62)
X_test : (6000, 62)
y_train: (24000,)
y_test : (6000,)


Identify Numeric and Categorical Columns

In [10]:
numeric_cols = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_cols = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numeric:", len(numeric_cols))
print("Categorical:", len(categorical_cols))

Numeric: 51
Categorical: 11


In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]),
            numeric_cols
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_cols
        )
    ]
)

print("Preprocessor created successfully")

Preprocessor created successfully


In [12]:
print(y_train.value_counts())

print("\nPercentages:\n")

print(
    (y_train.value_counts(normalize=True) * 100)
    .round(2)
)

academic_risk_band
Low         19512
Medium       3003
High         1353
Critical      132
Name: count, dtype: int64

Percentages:

academic_risk_band
Low         81.30
Medium      12.51
High         5.64
Critical     0.55
Name: proportion, dtype: float64


Build Logistic Regression Baseline

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

baseline_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=3000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

print("Baseline model created")

Baseline model created


Train

In [14]:
baseline_model.fit(X_train, y_train)

print("Training completed")

Training completed


c:\Users\aishw\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 3000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=3000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [15]:
preds = baseline_model.predict(X_test)

from sklearn.metrics import classification_report

print(classification_report(y_test, preds))

              precision    recall  f1-score   support

    Critical       0.45      0.76      0.56        33
        High       0.97      0.91      0.94       338
         Low       0.99      0.95      0.97      4878
      Medium       0.74      0.97      0.84       751

    accuracy                           0.95      6000
   macro avg       0.79      0.89      0.83      6000
weighted avg       0.96      0.95      0.95      6000



In [16]:
from sklearn.metrics import confusion_matrix
import pandas as pd

cm = confusion_matrix(y_test, preds)

cm_df = pd.DataFrame(
    cm,
    index=baseline_model.classes_,
    columns=baseline_model.classes_
)

print(cm_df)

          Critical  High   Low  Medium
Critical        25     8     0       0
High            31   307     0       0
Low              0     0  4621     257
Medium           0     0    25     726
